# Exploratory Data Analysis: Predicting Length of Stay (LOS) in ICU

## Project Overview
The objective of this project is to develop a machine learning pipeline to predict the Length of Stay (LOS) of patients admitted to the Intensive Care Unit (ICU). Predicting LOS is critical for hospital resource management, bed allocation, and identifying patients who may require long-term intensive care.

We will follow a traditional ML pipeline:

- Data Preprocessing & Cleaning

- Exploratory Data Analysis (EDA)

- Feature Engineering (Time-series windows)

- Model Training & Hyperparameter Tuning

- Performance Profiling & Evaluation

## The Dataset: MIMIC-III
The Medical Information Mart for Intensive Care (MIMIC-III) is a large, freely available database comprising de-identified health-related data associated with over forty thousand patients who stayed in critical care units of the Beth Israel Deaconess Medical Center between 2001 and 2012.

### Key Tables Involved:
- PATIENTS: Demographics (Gender, DOB, DOD).

- ADMISSIONS: Hospital stay information (Admission/Discharge times, Insurance, Ethnicity).

- CHARTEVENTS: The largest table. Contains all charted data for a patient (Heart rate, O2 saturation, etc.).

- ICUSTAYS: Defines the boundaries of a patient's stay in the ICU.

- D_ITEMS: Dictionary table to decode ITEMID from the events tables.

### Data Architecture and Connectivity
Understanding how the tables relate is crucial for feature engineering. The MIMIC-III database follows a hierarchical structure linked by specific identifiers:

#### Key Identifiers (The ID Hierarchy)
- SUBJECT_ID: Unique identifier for a specific patient.

- HADM_ID: Unique identifier for a single hospital admission. A patient can have multiple admissions over time.

- ICUSTAY_ID: Unique identifier for a single stay in the ICU. One hospital admission may involve multiple ICU transfers.

Relationship Logic: One Patient → Multiple Admissions → Multiple ICU Stays.
Each ICUSTAY_ID is strictly linked to one HADM_ID and one SUBJECT_ID.

#### Dataset Taxonomy
To navigate the 26 tables of MIMIC-III, we categorize them into four functional groups:

##### A. Patient Structural Tables (The "Skeleton")

These tables define the patient's journey and demographic background:

- PATIENTS: Basic demographics (Gender, DOB, Mortality).

- ADMISSIONS: Metadata for each hospital visit (Insurance, Ethnicity, Admission Type).

- ICUSTAYS: Critical for this project; contains entry/exit times and the calculated Length of Stay (LOS).

- TRANSFERS & SERVICES: Tracks movement within the hospital and the clinical departments responsible.

- CALLOUT: Logistics regarding the discharge process from the ICU.

##### B. Clinical ICU Data (The "Events")
This is where the high-resolution clinical data lives. These tables are time-stamped and contain the features for our ML model:

- CHARTEVENTS: The largest table (4.2GB compressed). Includes all charted observations (Vital signs, heart rate, oxygen levels).

- DATETIMEEVENTS: Significant time-stamped events (e.g., start of a procedure).

- INPUTEVENTS_CV / MV: Records of fluids and medications administered to the patient.

- OUTPUTEVENTS: Fluid outputs (e.g., urine, drainage).

- PROCEDUREEVENTS_MV: Procedures performed during the stay.

- NOTEEVENTS: Unstructured clinical notes from doctors and nurses.

- CAREGIVERS: Information about the staff recording the data.

##### C. Hospital System Data

Administrative and diagnostic records:

- LABEVENTS & MICROBIOLOGYEVENTS: Laboratory test results and cultures.

- PRESCRIPTIONS: Medications prescribed (pharmacy data).

- DIAGNOSES_ICD & PROCEDURES_ICD: Standardized medical codes (ICD) for billing and diagnosis.

##### D. Dictionary Tables (The "Translators")
Crucial for interpretability. In the clinical tables, variables are represented by ITEMID. To understand what an item is, we must join it with these dictionaries:

- D_ITEMS: Essential for this project. Maps ITEMID in CHARTEVENTS to human-readable labels (e.g., ITEMID 220045 → Heart Rate).

- D_LABITEMS: Decodes laboratory test codes.

- D_ICD_DIAGNOSES / PROCEDURES: Maps codes to actual medical diagnoses.


In [1]:
from config.config import MIMIC_DIR
from pyspark.sql import SparkSession
from pandas import DataFrame

# Configurando o Spark Session
spark = SparkSession.builder \
    .appName("MIMIC_III_LengthOfStay_Project") \
    .config("spark.sql.session.timeZone", "UTC") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .config("spark.sql.debug.maxToStringFields", 100) \
    .getOrCreate()

# Verificando se a sessão iniciou corretamente
print(f"Spark Session Initialized! Version: {spark.version}")

Spark Session Initialized! Version: 4.1.1


In [2]:
def create_df(path:str, describe:bool = True) -> DataFrame:
    # Carregando a tabela de pacientes
    df = spark.read.csv(path, header=True, inferSchema=True)

    if describe:
        df.show(5)
        df.printSchema()
        print(f"Number of rows: {df.count()}")
    return df

## Patients table

The PATIENTS table is the core demographic anchor of the MIMIC-III database. It defines every unique individual (SUBJECT_ID) who has been admitted to the hospital. With a total of 46,520 unique patients, this table contains "static" information—data that remains consistent throughout the patient's lifetime in the database.

Table Metadata
- Source: Derived from both CareVue and MetaVision ICU databases.
- Row Count: 46,520 rows (one per patient).
- Primary Key: SUBJECT_ID.

Column Definitions and Clinical Significance
- SUBJECT_ID: A unique identifier which specifies an individual patient. This is the candidate key for the table and is used to link the patient to all other clinical events across the database.

- GENDER: The genotypical sex of the patient (M/F). This serves as a fundamental baseline demographic feature for predictive modeling.

- DOB (Date of Birth): The date of birth of the given patient. While essential for calculating Age, it requires specific handling due to HIPAA de-identification rules for elderly patients.

- DOD (Date of Death): A merged date of death for the patient. This column combines data from the hospital (DOD_HOSP) and the Social Security database (DOD_SSN), giving priority to the hospital record if both exist.

- DOD_HOSP: The date of death as recorded specifically within the hospital database.

- DOD_SSN: The date of death obtained from the Social Security Administration death records. This is vital for identifying patients who passed away after being discharged from the hospital.

- EXPIRE_FLAG: A binary flag (0 or 1) indicating whether the patient has died. It is derived from the presence of a DOD. Note that this includes deaths both during and after a specific hospital stay.

*Preprocessing Rule: The 300-Year Shift*

The most important technical detail in this table is the handling of Protected Health Information (PHI) for elderly patients to comply with HIPAA regulations:

- The Problem: Patients older than 89 years at the time of their first admission have their DOB shifted to obscure their real age.

- The Process: The system determines the patient's age at their first admission and then sets the DOB to exactly 300 years before that date.

- The Impact: If you calculate age by simply subtracting DOB from ADMITTIME, these patients will appear to be roughly 300 years old.

- The Solution: During the preprocessing phase, we must detect these "century-old" outliers and cap their age (usually at 90 or the median of 91.4) to maintain model accuracy.

*Mortality and Feature Engineering*

While DOD and EXPIRE_FLAG are valuable for survival analysis, they must be used with caution for predicting Length of Stay (LOS). Since mortality is an outcome that occurs at the end or after a stay, including it as an input feature for an LOS predictor would result in Data Leakage (using information from the future to predict the present).

### Information that might interest
For the construction of the predictive model, the PATIENTS table will primarily contribute demographic data: *Gender and Age* (derived from the DOB column). Mortality-related variables, such as DOD and EXPIRE_FLAG, will be excluded from the training phase to prevent data leakage. This ensures the model only utilizes information that is realistically available at the moment of the patient's admission.

In [3]:
patients_df = create_df(f"{MIMIC_DIR}/PATIENTS.csv")

+------+----------+------+-------------------+-------------------+-------------------+-------+-----------+
|ROW_ID|SUBJECT_ID|GENDER|                DOB|                DOD|           DOD_HOSP|DOD_SSN|EXPIRE_FLAG|
+------+----------+------+-------------------+-------------------+-------------------+-------+-----------+
|   234|       249|     F|2075-03-13 00:00:00|               NULL|               NULL|   NULL|          0|
|   235|       250|     F|2164-12-27 00:00:00|2188-11-22 00:00:00|2188-11-22 00:00:00|   NULL|          1|
|   236|       251|     M|2090-03-15 00:00:00|               NULL|               NULL|   NULL|          0|
|   237|       252|     M|2078-03-06 00:00:00|               NULL|               NULL|   NULL|          0|
|   238|       253|     F|2089-11-26 00:00:00|               NULL|               NULL|   NULL|          0|
+------+----------+------+-------------------+-------------------+-------------------+-------+-----------+
only showing top 5 rows
root
 |-- ROW

## ADMISSIONS

The ADMISSIONS table serves as the primary record for every unique hospital visit. While the PATIENTS table defines who the individuals are, the ADMISSIONS table defines the circumstances, timing, and clinical context of each specific stay (HADM_ID). It is a foundational table for determining the "ground truth" of our target variable: the length of stay.

Table Metadata
- Source: Hospital admission, discharge, and transfer (ADT) database.

- Row Count: 58,976 rows.

- Primary Key: HADM_ID.

Links to: PATIENTS on SUBJECT_ID.

Column Definitions and Clinical Significance

- SUBJECT_ID & HADM_ID: HADM_ID is the unique identifier for a single admission. A single SUBJECT_ID may appear multiple times, representing readmissions.

- ADMITTIME & DISCHTIME: These provide the exact date and time of hospital admission and discharge. These are the most critical columns for calculating the total hospital stay duration.

- DEATHTIME: The time of in-hospital death. It is typically synchronized with DISCHTIME, though minor typographical discrepancies may exist in the raw data.

- ADMISSION_TYPE: Categorizes the visit as ELECTIVE, URGENT, NEWBORN, or EMERGENCY. Emergency and Urgent cases represent unplanned care and are major predictors of longer, more volatile stays compared to Elective (planned) surgeries.

- ADMISSION_LOCATION: Indicates where the patient was prior to arrival (e.g., Emergency Room, Clinic Referral, or Transfer from another facility). This provides context on the patient's clinical stability upon arrival.

- INSURANCE, LANGUAGE, RELIGION, MARITAL_STATUS, ETHNICITY: These demographic fields are captured at the time of each admission. Unlike the PATIENTS table, these values can change between different hospital visits for the same individual.

- EDREGTIME & EDOUTTIME: The registration and discharge times from the Emergency Department. The difference between these can indicate the "boarding time" before an ICU bed was made available.

- DIAGNOSIS: A free-text, preliminary diagnosis assigned by the admitting clinician. While informative, it is unstructured and not coded using standard ontologies like ICD-10 at this stage.

- HOSPITAL_EXPIRE_FLAG: A binary flag (0 or 1) indicating whether the patient died during this specific hospitalization.

- HAS_CHARTEVENTS_DATA: A crucial flag indicating if clinical measurements were recorded for this admission. Admissions with a 0 here likely lack the high-resolution data needed for deep analysis.

*Organ Donor Accounts Warning*

A specific technical quirk of the MIMIC-III database involves organ donor accounts. Accounts are sometimes created for patients who died in the hospital to facilitate organ donation, these records appear as distinct admissions but often have very short or even negative lengths of stay. These rows should be identified and filtered out during the data cleaning phase to avoid skewing the statistical distribution of stay durations.

*Text Consistency*

Except for the INSURANCE column, all text data in this table is stored in UPPER CASE. When performing string filtering or Natural Language Processing (NLP) on the DIAGNOSIS column, standardized casing must be applied.

### Information that might interest
For the construction of the predictive model, the ADMISSIONS table provides high-impact categorical features: Admission Type, Location. These serve as proxies for the severity of the patient's condition and the logistical complexity of their discharge. The ADMITTIME will be used in conjunction with the PATIENTS table to calculate the patient's age at the time of the event. To ensure data quality, only rows where HAS_CHARTEVENTS_DATA is equal to 1 will be considered for the final training set. Maybe we also should check about the use of the diagnosis column

In [4]:
admissions_df = create_df(f"{MIMIC_DIR}/ADMISSIONS.csv")

+------+----------+-------+-------------------+-------------------+---------+--------------+--------------------+--------------------+---------+--------+-----------------+--------------+---------+-------------------+-------------------+--------------------+--------------------+--------------------+
|ROW_ID|SUBJECT_ID|HADM_ID|          ADMITTIME|          DISCHTIME|DEATHTIME|ADMISSION_TYPE|  ADMISSION_LOCATION|  DISCHARGE_LOCATION|INSURANCE|LANGUAGE|         RELIGION|MARITAL_STATUS|ETHNICITY|          EDREGTIME|          EDOUTTIME|           DIAGNOSIS|HOSPITAL_EXPIRE_FLAG|HAS_CHARTEVENTS_DATA|
+------+----------+-------+-------------------+-------------------+---------+--------------+--------------------+--------------------+---------+--------+-----------------+--------------+---------+-------------------+-------------------+--------------------+--------------------+--------------------+
|    21|        22| 165315|2196-04-09 12:26:00|2196-04-10 15:54:00|     NULL|     EMERGENCY|EMERGENC

So, most admissions involve stays in the ICU:

In [5]:
admissions_df[admissions_df["HAS_CHARTEVENTS_DATA"] == 1].count()

57384

## ICUSTAYS

The ICUSTAYS table defines each unique stay in the Intensive Care Unit (ICU). While a patient is admitted to the hospital (HADM_ID), they may be transferred in and out of the ICU multiple times. This table is the most specific level of the patient stay hierarchy and contains the primary target variable for our predictive model: the ICU Length of Stay (LOS).

Table Metadata
- Source: Derived from the hospital's TRANSFERS table.

- Row Count: 61,532 rows.

- Primary Key: ICUSTAY_ID.

Links to: PATIENTS on SUBJECT_ID, ADMISSIONS on HADM_ID.

Column Definitions and Clinical Significance
- SUBJECT_ID, HADM_ID & ICUSTAY_ID: Identifiers that specify the patient's hierarchy. ICUSTAY_ID is a generated identifier (not found in raw hospital data) used to group ICU transfers that occur within 24 hours of each other.

- DBSOURCE: Indicates the original ICU information system: 'carevue' (2001-2008) or 'metavision' (2008-2012). This is important because data archiving and item codes often differ between these two systems.

- FIRST_CAREUNIT & LAST_CAREUNIT: The first and last type of ICU where the patient was treated (e.g., MICU for Medical, SICU for Surgical, CCU for Coronary Care). The type of unit can be a strong indicator of the patient's condition severity.

- FIRST_WARDID & LAST_WARDID: Technical identifiers for the physical ICU units (wards) where the patient stayed.

- INTIME & OUTTIME: The exact date and time the patient entered and exited the ICU. These timestamps define the boundaries for our clinical observation windows.

- LOS (Length of Stay): The total duration of the ICU stay measured in fractional days. This is the ground truth variable we aim to predict.

Technical Considerations
Transfer Grouping: If a patient is transferred between different ICU units within a 24-hour window, these movements are grouped under a single ICUSTAY_ID.

*Clinical Context: Care Units*
The type of ICU (FIRST_CAREUNIT) provides critical context for Length of Stay:

- NICU (Neonatal): Stays are often longer and follow different clinical patterns.

- CCU: Coronary Care Unit.

- CSRU (Cardiac Surgery): Stays are usually shorter and highly protocol-driven.

- MICU/SICU (Medical/Surgical): Stays can vary significantly based on the complexity of the acute illness.

### Information that might interest
For the construction of the predictive model, the ICUSTAYS table provides our target variable: LOS. We will use INTIME to define the "starting point" of our prediction (e.g., using data from the first 24 hours after INTIME to predict the total LOS). Furthermore, the FIRST_CAREUNIT will be used as a categorical feature to account for the different clinical workflows and average stay durations associated with different specialized ICU departments.

In [6]:
icustays_df = create_df(f"{MIMIC_DIR}/ICUSTAYS.csv")

+------+----------+-------+----------+--------+--------------+-------------+------------+-----------+-------------------+-------------------+------+
|ROW_ID|SUBJECT_ID|HADM_ID|ICUSTAY_ID|DBSOURCE|FIRST_CAREUNIT|LAST_CAREUNIT|FIRST_WARDID|LAST_WARDID|             INTIME|            OUTTIME|   LOS|
+------+----------+-------+----------+--------+--------------+-------------+------------+-----------+-------------------+-------------------+------+
|   365|       268| 110404|    280836| carevue|          MICU|         MICU|          52|         52|2198-02-14 23:27:38|2198-02-18 05:26:11| 3.249|
|   366|       269| 106296|    206613| carevue|          MICU|         MICU|          52|         52|2170-11-05 11:05:29|2170-11-08 17:46:57|3.2788|
|   367|       270| 188028|    220345| carevue|           CCU|          CCU|          57|         57|2128-06-24 15:05:20|2128-06-27 12:32:29|2.8939|
|   368|       271| 173727|    249196| carevue|          MICU|         SICU|          52|         23|2120-

## Transfer

The TRANSFERS table tracks every movement of a patient between different physical locations (wards) and care units. It records when a patient is admitted, moved to a different specialty unit, or finally discharged. This table is essential for understanding the administrative flow and the specific sequence of care units a patient visited.

Table Metadata
- Source: Hospital admission, discharge, and transfer (ADT) database.

- Row Count: 261,897 rows.

- Primary Key: ROW_ID.

Links to: PATIENTS on SUBJECT_ID, ADMISSIONS on HADM_ID, and ICUSTAYS on ICUSTAY_ID.

Column Definitions and Clinical Significance
- SUBJECT_ID, HADM_ID & ICUSTAY_ID: standard identifiers. Note that ICUSTAY_ID will be null for transfers that occur in non-ICU wards (like a regular hospital floor).

- EVENTTYPE: Describes the nature of the movement: 'ADMIT' (entering the hospital), 'TRANSFER' (moving between wards), or 'DISCHARGE' (leaving the hospital).

- PREV_CAREUNIT & CURR_CAREUNIT: These columns identify the clinical specialty of the unit (e.g., MICU, SICU, CCU). If the location is not an ICU, these fields are usually null.

- PREV_WARDID & CURR_WARDID: The physical ID of the hospital ward. Every ICU is a ward, but not every ward is an ICU.

- INTIME & OUTTIME: The precise timestamps for when the patient entered and left the current ward (CURR_WARDID).

- LOS (Length of Stay): The duration of stay specifically within that single ward, measured in fractional days.


*Technical Considerations: The Origin of ICUSTAYS*

- Hierarchy: The ICUSTAYS table is actually a "collapsed" version of this table. When the system detects multiple transfer rows with ICU ward IDs that are less than 24 hours apart, it groups them into a single ICUSTAY_ID.

*Information that might interest*

Maybe we need this chart to monitor the patient in the ICU. To see which units he has been assigned to.

In [7]:
transfers_df = create_df(f"{MIMIC_DIR}/TRANSFERS.csv")

+------+----------+-------+----------+----------+---------+-------------+-------------+-----------+-----------+-------------------+-------------------+------+
|ROW_ID|SUBJECT_ID|HADM_ID|ICUSTAY_ID|  DBSOURCE|EVENTTYPE|PREV_CAREUNIT|CURR_CAREUNIT|PREV_WARDID|CURR_WARDID|             INTIME|            OUTTIME|   LOS|
+------+----------+-------+----------+----------+---------+-------------+-------------+-----------+-----------+-------------------+-------------------+------+
|   657|       111| 192123|    254245|   carevue| transfer|          CCU|         MICU|          7|         23|2142-04-29 15:27:11|2142-05-04 20:38:33|125.19|
|   658|       111| 192123|      NULL|   carevue| transfer|         MICU|         NULL|         23|         45|2142-05-04 20:38:33|2142-05-05 11:46:32| 15.13|
|   659|       111| 192123|      NULL|   carevue|discharge|         NULL|         NULL|         45|       NULL|2142-05-05 11:46:32|               NULL|  NULL|
|   660|       111| 155897|    249202|metavisi

## Services

The SERVICES table provides a record of the medical teams (services) that were responsible for a patient during their hospital stay. While the TRANSFERS table tracks physical locations, the SERVICES table tracks clinical responsibility. A patient may be physically located in the MICU (Medical ICU) but be under the care of the NSURG (Neurosurgical) service.

Table Metadata
- Source: Hospital database.

- Row Count: 73,343 rows.

- Primary Key: ROW_ID.

Links to: PATIENTS on SUBJECT_ID, ADMISSIONS on HADM_ID.

Column Definitions and Clinical Significance
- SUBJECT_ID & HADM_ID: Standard identifiers to link the service record to the specific patient and hospital admission.

- TRANSFERTIME: The exact timestamp when the patient moved from the previous service to the current one.

- PREV_SERVICE & CURR_SERVICE: The clinical services the patient was under. These are stored as abbreviations (e.g., MED, SURG, TRAUM).

*Service Types and Abbreviations*
The table uses specific codes to define the type of care. Understanding these is vital for patient stratification:

- Medical Services: MED (General Medicine), CMED (Cardiac Medical), NMED (Neurologic Medical), OMED (Oncologic Medical).

- Surgical Services: SURG (General Surgery), CSURG (Cardiac Surgery), NSURG (Neurosurgical), TSURG (Thoracic), VSURGV (Vascular).

- Specialized Services: TRAUM (Trauma), PSYCH (Psychiatric), OBS (Obstetrics), GYN (Gynecology).

- Newborns: NB and NBB.

Technical Considerations: Service vs. Location
- The "Bed Shortage" Factor: A patient might be located in a specific ICU due to bed availability, but the SERVICES table identifies the actual clinical specialty managing their case.

- Identifying Patient Cohorts: To accurately identify "surgical patients" or "cardiac patients" for predictive modeling, it is recommended to use this table rather than relying solely on the physical ICU type.

*Information that might interest*

For the construction of the predictive model, the SERVICES table is essential for identifying the type of service a patient is receiving. The clinical service is often a better predictor of LOS (Length of Stay) than the physical ward, as surgical patients (SURG) and trauma patients (TRAUM) have fundamentally different recovery trajectories and discharge requirements. We can use this table to create a "Surgical vs. Medical" binary feature or a categorical "Primary Service" feature to refine our predictions. (PREV_SERVICE, CURR_SERVICE)

In [8]:
services_df = create_df(f"{MIMIC_DIR}/SERVICES.csv")

+------+----------+-------+-------------------+------------+------------+
|ROW_ID|SUBJECT_ID|HADM_ID|       TRANSFERTIME|PREV_SERVICE|CURR_SERVICE|
+------+----------+-------+-------------------+------------+------------+
|   758|       471| 135879|2122-07-22 14:07:27|       TSURG|         MED|
|   759|       471| 135879|2122-07-26 18:31:49|         MED|       TSURG|
|   760|       472| 173064|2172-09-28 19:22:15|        NULL|        CMED|
|   761|       473| 129194|2201-01-09 20:16:45|        NULL|          NB|
|   762|       474| 194246|2181-03-23 08:24:41|        NULL|          NB|
+------+----------+-------+-------------------+------------+------------+
only showing top 5 rows
root
 |-- ROW_ID: integer (nullable = true)
 |-- SUBJECT_ID: integer (nullable = true)
 |-- HADM_ID: integer (nullable = true)
 |-- TRANSFERTIME: timestamp (nullable = true)
 |-- PREV_SERVICE: string (nullable = true)
 |-- CURR_SERVICE: string (nullable = true)

Number of rows: 73343


## CALLOUT

The CALLOUT table provides critical information regarding the ICU discharge planning process. It records the specific moment a medical provider deems a patient "ready for discharge" (the call out) versus the moment the patient actually leaves the unit. This gap often represents administrative or logistical delays (such as waiting for a ward bed) rather than clinical necessity.

Table Metadata
- Source: Hospital database.

- Row Count: 34,499 rows.

- Primary Key: ROW_ID.

Links to: PATIENTS on SUBJECT_ID, ADMISSIONS on HADM_ID.

Column Definitions and Clinical Significance
- SUBJECT_ID & HADM_ID: Standard identifiers to link the discharge request to the patient and admission.

- CREATETIME: The exact timestamp when the provider registered that the patient was ready to leave the ICU.

- OUTCOMETIME: The timestamp when the final outcome (discharge or cancellation) occurred.

- CALLOUT_STATUS & CALLOUT_OUTCOME: 'Discharged' indicates the patient successfully left the ICU, while 'Cancelled' means the discharge request was retracted (possibly due to clinical deterioration).

- CALLOUT_SERVICE: The medical service that will take over the patient's care after they leave the ICU.

- REQUEST_TELE, RESP, CDIFF, MRSA, VRE: Binary indicators for specialized precautions required in the next ward (e.g., telemetry monitoring or infection control for MRSA/VRE).

- ACKNOWLEDGETIME: When a coordinator acknowledged the request and began searching for a destination bed.

*Technical Considerations: The "Ready-to-Leave" Gap*
Data Coverage: This table does not cover all patients. Data collection began midway through the MIMIC project, and it is never available for neonates (NICU).

Efficiency Metric: The difference between CREATETIME and OUTCOMETIME is a powerful metric for hospital efficiency. It reveals how much of the LOS (Length of Stay) was due to clinical care versus boarding time (waiting for a bed).

Discharge Destinations
- CALLOUT_WARDID = 0: Indicates the patient is being discharged directly to Home.

- CALLOUT_WARDID = 1: Indicates "First available ward," showing an urgent need to clear the ICU bed.

*Information that might interest*

I don't think this table is so important in this case

In [9]:
callout_df = create_df(f"{MIMIC_DIR}/CALLOUT.csv")

+------+----------+-------+-------------+---------------+-----------+-------------+--------------+---------------+------------+------------+-------------+------------+-----------+--------------+---------------+----------------+------------------+-------------------+-------------------+-------------------+-------------------+--------------------+----------------------+
|ROW_ID|SUBJECT_ID|HADM_ID|SUBMIT_WARDID|SUBMIT_CAREUNIT|CURR_WARDID|CURR_CAREUNIT|CALLOUT_WARDID|CALLOUT_SERVICE|REQUEST_TELE|REQUEST_RESP|REQUEST_CDIFF|REQUEST_MRSA|REQUEST_VRE|CALLOUT_STATUS|CALLOUT_OUTCOME|DISCHARGE_WARDID|ACKNOWLEDGE_STATUS|         CREATETIME|         UPDATETIME|    ACKNOWLEDGETIME|        OUTCOMETIME|FIRSTRESERVATIONTIME|CURRENTRESERVATIONTIME|
+------+----------+-------+-------------+---------------+-----------+-------------+--------------+---------------+------------+------------+-------------+------------+-----------+--------------+---------------+----------------+------------------+------------

## CHARTEVENTS

The CHARTEVENTS table is the largest and most information-dense table in the MIMIC-III database. It serves as the primary repository for all "charted" clinical data, effectively representing the patient’s electronic medical record (EMR). This includes routine vital signs, ventilator settings, mental status (GCS), and even laboratory values that were copied into the bedside chart for clinician review.

Table Metadata
- Source: CareVue and MetaVision ICU databases.

- Row Count: 330,712,483 rows.

- Primary Key: ROW_ID.

Links to: PATIENTS (SUBJECT_ID), ADMISSIONS (HADM_ID), ICUSTAYS (ICUSTAY_ID), and D_ITEMS (ITEMID).

Column Definitions and Clinical Significance
- ITEMID: The identifier for the specific type of measurement. To understand what is being measured (e.g., Heart Rate vs. Oxygen Saturation), this must be joined with the D_ITEMS table.

- CHARTTIME & STORETIME: CHARTTIME is the timestamp when the measurement actually occurred (the most important for time-series analysis). STORETIME is when a staff member manually validated or entered the data.

- VALUE & VALUENUM: VALUE contains the raw measurement as text. If the data is numeric, VALUENUM stores it as a float for easier computation. For clinical scores like the Glasgow Coma Scale, VALUENUM holds the numerical score while VALUE might include descriptive text.

- VALUEUOM: The Unit of Measurement (e.g., bpm, mmHg, °C).

- CGID: The identifier for the caregiver (nurse, doctor, therapist) who recorded or validated the entry.

Technical Considerations: Handling Big Data
Scale: Due to its massive size (330M+ rows), this table cannot be processed using standard libraries like Pandas on a single machine. It requires distributed computing (PySpark) and aggressive filtering.

Redundancy: Lab values found here are often duplicates of the LABEVENTS table. Per documentation guidelines, if values disagree, LABEVENTS should be treated as the "ground truth."

System Differences:

- MetaVision: Provides WARNING and ERROR flags for data quality.

- CareVue: Uses RESULTSTATUS and STOPPED to indicate if a measurement was manual or automatic.

*Feature Engineering: The Observation Window*

For predicting LOS (Length of Stay), it is common practice to extract features from this table within a specific "Observation Window" (e.g., the first 24 or 48 hours of an ICU stay). This allows the model to learn from the patient's initial physiological stability without using data from the future.

*Information that might interest*

For the construction of the predictive model, CHARTEVENTS is the most critical source for physiological features. Vital signs (Heart Rate, Blood Pressure, Temperature) and clinical scores (GCS) extracted from this table are high-frequency indicators of a patient's severity of illness. Because this table is massive, we will focus on a subset of ITEMIDs (defined in D_ITEMS) that are most clinically relevant to stay duration, such as respiratory support levels and hemodynamic stability indicators.

In [10]:
chartevents_df = create_df(f"{MIMIC_DIR}/CHARTEVENTS.csv")

+------+----------+-------+----------+------+-------------------+-------------------+-----+-----+--------+--------+-------+-----+------------+-------+
|ROW_ID|SUBJECT_ID|HADM_ID|ICUSTAY_ID|ITEMID|          CHARTTIME|          STORETIME| CGID|VALUE|VALUENUM|VALUEUOM|WARNING|ERROR|RESULTSTATUS|STOPPED|
+------+----------+-------+----------+------+-------------------+-------------------+-----+-----+--------+--------+-------+-----+------------+-------+
|   788|        36| 165660|    241249|223834|2134-05-12 12:00:00|2134-05-12 13:56:00|17525|   15|    15.0|   L/min|      0|    0|        NULL|   NULL|
|   789|        36| 165660|    241249|223835|2134-05-12 12:00:00|2134-05-12 13:56:00|17525|  100|   100.0|    NULL|      0|    0|        NULL|   NULL|
|   790|        36| 165660|    241249|224328|2134-05-12 12:00:00|2134-05-12 12:18:00|20823|  .37|    0.37|    NULL|      0|    0|        NULL|   NULL|
|   791|        36| 165660|    241249|224329|2134-05-12 12:00:00|2134-05-12 12:19:00|20823|   

## DATETIMEEVENTS

The DATETIMEEVENTS table contains all clinical observations that are recorded as dates or timestamps. Unlike CHARTEVENTS, which stores numerical or text values (like heart rate or blood pressure), this table stores time-based events such as the "date of last dialysis" or "time of last dressing change."

Table Metadata
- Source: CareVue and MetaVision ICU databases.

- Row Count: 4,485,937 rows.

- Primary Key: ROW_ID.

Links to: PATIENTS (SUBJECT_ID), ADMISSIONS (HADM_ID), ICUSTAYS (ICUSTAY_ID), and D_ITEMS (ITEMID).

Column Definitions and Clinical Significance
- ITEMID: Identifies the specific date-based event being recorded. To know exactly what the date refers to, this must be joined with the D_ITEMS table.

- CHARTTIME & STORETIME: CHARTTIME is the time the observation was charted (the actual event time), while STORETIME is when it was manually validated in the system.

- VALUE: This is the most important column in this table. It contains the actual date/time value of the measurement (e.g., the specific time a patient was last extubated).

- VALUEUOM: The unit of measurement, which for this table is typically a date format string.

- WARNING, ERROR, RESULTSTATUS, STOPPED: System-specific flags from CareVue and MetaVision used for data quality and status tracking.

Technical Considerations: Anonymization and Chronology
- Date Shifting: In compliance with HIPAA, all dates in this table have been shifted to protect patient confidentiality.

- Chronological Integrity: Although the dates themselves are not "real," the chronology is preserved. The time difference between two dates (e.g., the time between two dialysis sessions) remains accurate and true to the original clinical reality.

Clinical Context: Procedures and Milestones

This table is essential for tracking clinical milestones that don't fit into regular "vital sign" monitoring. Examples include:

- Last Dialysis: Indicates renal replacement therapy frequency.

- Extubation Time: A critical marker for respiratory recovery.

- Procedure Times: When specific invasive interventions occurred.

*Information that might interest*

For the construction of the predictive model, DATETIMEEVENTS is a key source for calculating time-delta features. For instance, the time elapsed since the last major procedure or the frequency of specific interventions can be strong predictors of a patient's stability and their likely LOS (Length of Stay). We can use these timestamps to determine if a patient is in a "recovery phase" (e.g., recently extubated) or a "critical phase," providing the model with a better sense of the patient's clinical trajectory.

In [11]:
datetimeevents_df = create_df(f"{MIMIC_DIR}/DATETIMEEVENTS.csv")

+------+----------+-------+----------+------+-------------------+-------------------+-----+-----+--------+-------+-----+------------+--------+
|ROW_ID|SUBJECT_ID|HADM_ID|ICUSTAY_ID|ITEMID|          CHARTTIME|          STORETIME| CGID|VALUE|VALUEUOM|WARNING|ERROR|RESULTSTATUS| STOPPED|
+------+----------+-------+----------+------+-------------------+-------------------+-----+-----+--------+-------+-----+------------+--------+
|   711|      7657| 121183|    297945|  3411|2172-03-14 11:00:00|2172-03-14 11:52:00|16446| NULL|    Date|   NULL| NULL|        NULL|NotStopd|
|   712|      7657| 121183|    297945|  3411|2172-03-14 13:00:00|2172-03-14 12:36:00|16446| NULL|    Date|   NULL| NULL|        NULL|NotStopd|
|   713|      7657| 121183|    297945|  3411|2172-03-14 15:00:00|2172-03-14 15:10:00|14957| NULL|    Date|   NULL| NULL|        NULL|NotStopd|
|   714|      7657| 121183|    297945|  3411|2172-03-14 17:00:00|2172-03-14 17:01:00|16446| NULL|    Date|   NULL| NULL|        NULL|NotStopd|

## INPUTEVENTS_CV

The INPUTEVENTS_CV table contains all intake data for patients whose care was managed via the CareVue clinical information system (typically admissions between 2001 and 2008). It records the administration of intravenous medications, fluids, and specialized solutions. This table is essential for tracking the volume and rate of life-sustaining treatments like vasopressors or volume resuscitations.

Table Metadata
- Source: CareVue ICU database.

- Row Count: 17,527,935 rows.

- Primary Key: ROW_ID.

Links to: PATIENTS (SUBJECT_ID), ADMISSIONS (HADM_ID), ICUSTAYS (ICUSTAY_ID), and D_ITEMS (ITEMID).

Column Definitions and Clinical Significance
- ITEMID: Identifier for the substance administered.

- CHARTTIME: * For Amounts (volumes): Represents the "end time" (when the volume was finished being received).
For Rates (continuous infusions): Represents the "start time" (when that specific rate was set).

- AMOUNT & AMOUNTUOM: The total quantity of the substance administered (e.g., 500 mL of Saline).

- RATE & RATEUOM: The speed at which a drug was administered (e.g., mcg/kg/min for vasopressors).

- ORDERID & LINKORDERID: * ORDERID: Groups different items in the same solution (e.g., Noradrenaline mixed in Normal Saline).

- LINKORDERID: Connects the same order across changes, such as a rate adjustment for the same infusion.

- STOPPED & NEWBOTTLE: Indicators of whether an infusion was disconnected or if a fresh bag/bottle was hung at the bedside.

*Clinical Context: Fluid and Drug Management*

Tracking inputs is a direct proxy for clinical instability:

- Vasopressors: High rates or frequent adjustments usually indicate hemodynamic instability.

- Fluid Balance: Comparing total inputs from this table against outputs (from OUTPUTEVENTS) is a standard clinical practice to assess kidney function and fluid overload.

*Information that might interest* 

For the construction of the predictive model, INPUTEVENTS_CV provides features related to treatment intensity. A patient requiring multiple continuous infusions (high number of active LINKORDERIDs) or high doses of vasopressors is likely to have a more guarded prognosis and a longer LOS (Length of Stay). We can use this table to engineer features such as "Total Fluid Intake in first 24h" or "Presence of Vasopressors," which are strong indicators of the level of care required.

In [12]:
inputevents_cv_df = create_df(f"{MIMIC_DIR}/INPUTEVENTS_CV.csv")

+------+----------+-------+----------+-------------------+------+------+---------+----+-------+-------------------+-----+-------+-----------+-------+---------+--------------+-----------------+-------------+------------+---------------+------------+
|ROW_ID|SUBJECT_ID|HADM_ID|ICUSTAY_ID|          CHARTTIME|ITEMID|AMOUNT|AMOUNTUOM|RATE|RATEUOM|          STORETIME| CGID|ORDERID|LINKORDERID|STOPPED|NEWBOTTLE|ORIGINALAMOUNT|ORIGINALAMOUNTUOM|ORIGINALROUTE|ORIGINALRATE|ORIGINALRATEUOM|ORIGINALSITE|
+------+----------+-------+----------+-------------------+------+------+---------+----+-------+-------------------+-----+-------+-----------+-------+---------+--------------+-----------------+-------------+------------+---------------+------------+
|   592|     24457| 184834|    205776|2193-09-11 09:00:00| 30056| 100.0|       ml|NULL|   NULL|2193-09-11 11:12:00|14990| 756654|    9359133|   NULL|     NULL|          NULL|               ml|         Oral|        NULL|           NULL|        NULL|
|   

## INPUTEVENTS_MV

The INPUTEVENTS_MV table contains all intake/input data for patients managed via the MetaVision clinical information system. It is the modern counterpart to INPUTEVENTS_CV and provides a more structured recording of intravenous medications, nutrition, and fluid boluses. A key feature of this table is the explicit use of start and end timestamps for every event.

Table Metadata

- Source: MetaVision ICU database (2008–2012).

- Row Count: 3,618,991 rows.

- Primary Key: ROW_ID.

Links to: PATIENTS (SUBJECT_ID), ADMISSIONS (HADM_ID), ICUSTAYS (ICUSTAY_ID), and D_ITEMS (ITEMID).

Column Definitions and Clinical Significance
- STARTTIME & ENDTIME: Define the exact duration of an administration. For boluses (rapid injections), the EN DTIME is typically set to one minute after the STARTTIME.

- ITEMID: Identifier for the substance. In MetaVision, all ITEMID values are above 220,000.

- AMOUNT & RATE: The quantity and speed of administration. TOTALAMOUNT reflects the full volume of the bag being hung.

- ORDERID & LINKORDERID: Used to group components of a single solution and track changes (like rate adjustments) across a continuous infusion.

- STATUSDESCRIPTION: Provides crucial context on why a row ended (e.g., 'Changed' for rate updates, 'FinishedRunning' for empty bags, or 'Stopped' by a nurse).

- PATIENTWEIGHT: The weight of the patient at the time of the order, used for weight-based dosing (mcg/kg/min).

Original vs. True Rate: The table distinguishes between ORIGINALRATE (the planned dose) and RATE (what was actually delivered). While usually identical, discrepancies can occur if a caregiver manually "flushes" the remaining fluid.

*Clinical Context: Dosing and Severity* 

The metadata provided in ORDERCATEGORYNAME and ORDERCOMPONENTTYPEDESCRIPTION allows for easier filtering of medications. For instance, you can easily distinguish between a "Main Order" (the drug) and an "Additive" (like electrolytes added to a carrier fluid).

*Information that might interest*

For the construction of the predictive model, INPUTEVENTS_MV is a high-quality source for dynamic features. Because it includes PATIENTWEIGHT, we can normalize drug dosages across different patients. The presence of specific STATUSDESCRIPTION values like 'Paused' or 'Stopped' can also serve as a proxy for a patient's improving condition (weaning off vasopressors), which is a strong indicator of a shorter remaining LOS (Length of Stay).

In [13]:
inputevents_mv_df = create_df(f"{MIMIC_DIR}/INPUTEVENTS_MV.csv")

+------+----------+-------+----------+-------------------+-------------------+------+-----------+---------+---------+-------+-------------------+-----+-------+-----------+--------------------+--------------------------+-----------------------------+------------------------+-------------+-----------+--------------+---------+------------------+------------+-----------------+-----------------+-------------------+-------------------+--------------+------------+
|ROW_ID|SUBJECT_ID|HADM_ID|ICUSTAY_ID|          STARTTIME|            ENDTIME|ITEMID|     AMOUNT|AMOUNTUOM|     RATE|RATEUOM|          STORETIME| CGID|ORDERID|LINKORDERID|   ORDERCATEGORYNAME|SECONDARYORDERCATEGORYNAME|ORDERCOMPONENTTYPEDESCRIPTION|ORDERCATEGORYDESCRIPTION|PATIENTWEIGHT|TOTALAMOUNT|TOTALAMOUNTUOM|ISOPENBAG|CONTINUEINNEXTDEPT|CANCELREASON|STATUSDESCRIPTION|COMMENTS_EDITEDBY|COMMENTS_CANCELEDBY|      COMMENTS_DATE|ORIGINALAMOUNT|ORIGINALRATE|
+------+----------+-------+----------+-------------------+------------------

## OUTPUTEVENTS

The OUTPUTEVENTS table contains all "output" data for patients, representing substances eliminated or drained from the body. This primarily includes urine output, but also encompasses chest tube drainage, gastric secretions, and other bodily fluids. Monitoring these outputs is a standard clinical method for assessing organ perfusion and fluid balance.

Table Metadata
Source: CareVue and MetaVision ICU databases.

Row Count: 4,349,218 rows.

Primary Key: ROW_ID.

Links to: PATIENTS (SUBJECT_ID), ADMISSIONS (HADM_ID), ICUSTAYS (ICUSTAY_ID), and D_ITEMS (ITEMID).

Column Definitions and Clinical Significance
- ITEMID: Identifier for the type of output (e.g., Foley catheter urine, voided urine, or surgical drain). MetaVision IDs are >220,000, while CareVue IDs for outputs are generally between 40,000–49,999.

- CHARTTIME: The time the output was recorded. For periodic measurements (like hourly urine), this represents the end of the collection interval.

- VALUE & VALUEUOM: The volume or amount of the substance (usually in mL).

- STORETIME: The time the measurement was manually entered or validated by the clinical staff.

- ISERROR (MetaVision Only): A binary flag indicating if a caregiver marked the entry as a recording error.

- STOPPED & NEWBOTTLE: Indicators used primarily for drainage systems to show when a collection bag was emptied or a device was disconnected.

Urine Output (UO): UO is one of the most frequently used features in predictive modeling for ICU patients, as it is a core component of acuity scores like SOFA or SAPS II.

*Clinical Context: Renal and Respiratory Health*

- Low Urine Output (Oliguria): A strong early warning sign of Acute Kidney Injury (AKI) and overall circulatory shock.

- Drainage Volumes: High output from surgical or chest drains can indicate internal bleeding or the need for a return to the operating room.

*Information that might interest*

For the construction of the predictive model, OUTPUTEVENTS provides vital indicators of physiological recovery. A stable or increasing urine output in the first 24 hours is often a positive sign, correlating with a shorter LOS (Length of Stay). Conversely, patients requiring aggressive fluid resuscitation with minimal output are at higher risk for complications, likely resulting in a prolonged ICU stay. We can engineer features such as "Total Urine Output per Hour" or "Cumulative 24h Fluid Balance" to serve as high-impact predictors.a

In [14]:
outputevents_df = create_df(f"{MIMIC_DIR}/OUTPUTEVENTS.csv")

+------+----------+-------+----------+-------------------+------+-----+--------+-------------------+-----+-------+---------+-------+
|ROW_ID|SUBJECT_ID|HADM_ID|ICUSTAY_ID|          CHARTTIME|ITEMID|VALUE|VALUEUOM|          STORETIME| CGID|STOPPED|NEWBOTTLE|ISERROR|
+------+----------+-------+----------+-------------------+------+-----+--------+-------------------+-----+-------+---------+-------+
|   344|     21219| 177991|    225765|2142-09-08 10:00:00| 40055|200.0|      ml|2142-09-08 12:08:00|17269|   NULL|     NULL|   NULL|
|   345|     21219| 177991|    225765|2142-09-08 12:00:00| 40055|200.0|      ml|2142-09-08 12:08:00|17269|   NULL|     NULL|   NULL|
|   346|     21219| 177991|    225765|2142-09-08 13:00:00| 40055|120.0|      ml|2142-09-08 13:39:00|17269|   NULL|     NULL|   NULL|
|   347|     21219| 177991|    225765|2142-09-08 14:00:00| 40055|100.0|      ml|2142-09-08 16:17:00|17269|   NULL|     NULL|   NULL|
|   348|     21219| 177991|    225765|2142-09-08 16:00:00| 40055|200.

## PROCEDUREEVENTS_MV

The PROCEDUREEVENTS_MV table records all clinical procedures performed on patients managed via the MetaVision system. Unlike static diagnosis codes, this table captures the duration of active interventions. It tracks everything from major procedures like continuous renal replacement therapy (CRRT) to routine tasks like peripheral IV insertion, providing a clear timeline of the intensity of care.

Table Metadata
- Source: MetaVision ICU database (2008–2012).

- Row Count: 258,066 rows.

Primary Key: ROW_ID.

Links to: PATIENTS (SUBJECT_ID), ADMISSIONS (HADM_ID), ICUSTAYS (ICUSTAY_ID), and D_ITEMS (ITEMID).

Column Definitions and Clinical Significance
STARTTIME & ENDTIME: These define the exact window during which the procedure took place. For continuous procedures (like mechanical ventilation), the difference between these two timestamps provides the total duration of the intervention.

ITEMID: Identifier for the procedure. Common procedures include "Ventilation," "Dialysis," and "Central Line Insertion."

VALUE & VALUEUOM: Usually represents the count or amount associated with the procedure, if applicable.

LOCATION & LOCATIONCATEGORY: Specifies where the procedure was performed (e.g., at the bedside or in the operating room).

ORDERCATEGORYNAME: Categorizes the procedure, making it easier to filter for specific types of care (e.g., "Intubation/Extubation" or "Invasive Lines").

STATUSDESCRIPTION: Indicates the state of the procedure (e.g., 'FinishedRunning', 'Stopped', or 'Cancelled').

Technical Considerations: Procedure Duration
Continuous vs. Point Events: Some procedures are instantaneous (like a single X-ray), while others are continuous. For the latter, calculating the duration (ENDTIME - STARTTIME) is a standard feature engineering step for LOS prediction.

MetaVision Exclusivity: This table only contains data for MetaVision patients. For older CareVue patients, similar information is often found within the CHARTEVENTS or INPUTEVENTS_CV tables under specific ITEMIDs.

Clinical Context: Intervention Intensity
Procedures are one of the strongest proxies for a patient's acuity:

Mechanical Ventilation: One of the most significant drivers of ICU LOS.

Renal Replacement (Dialysis): Indicates severe organ failure and a high likelihood of a prolonged stay.

Invasive Monitoring: The need for arterial lines or central venous pressure monitoring reflects hemodynamic instability.

Information that might interest
For the construction of the predictive model, PROCEDUREEVENTS_MV provides high-impact temporal features. The presence and duration of invasive procedures are among the strongest predictors of Length of Stay (LOS). We can use this table to create features such as "Total hours on ventilator in the first 24h" or "Number of invasive procedures initiated." These features allow the model to understand not just what the patient has (diagnosis), but what is being done to them (treatment intensity).

In [15]:
procedureevents_mv_df = create_df(f"{MIMIC_DIR}/PROCEDUREEVENTS_MV.csv")

+------+----------+-------+----------+-------------------+-------------------+------+------+--------+--------+----------------+-------------------+-----+-------+-----------+-----------------+--------------------------+------------------------+---------+------------------+------------+-----------------+-----------------+-------------------+-------------+
|ROW_ID|SUBJECT_ID|HADM_ID|ICUSTAY_ID|          STARTTIME|            ENDTIME|ITEMID| VALUE|VALUEUOM|LOCATION|LOCATIONCATEGORY|          STORETIME| CGID|ORDERID|LINKORDERID|ORDERCATEGORYNAME|SECONDARYORDERCATEGORYNAME|ORDERCATEGORYDESCRIPTION|ISOPENBAG|CONTINUEINNEXTDEPT|CANCELREASON|STATUSDESCRIPTION|COMMENTS_EDITEDBY|COMMENTS_CANCELEDBY|COMMENTS_DATE|
+------+----------+-------+----------+-------------------+-------------------+------+------+--------+--------+----------------+-------------------+-----+-------+-----------+-----------------+--------------------------+------------------------+---------+------------------+------------+---

## NOTEEVENTS

The NOTEEVENTS table contains all clinical notes and reports recorded for patients. This includes nursing notes, physician progress notes, radiology reports, ECG interpretations, and discharge summaries. It represents the richest source of qualitative data in the database, capturing clinical reasoning, family discussions, and nuanced observations that numerical data cannot reflect.

Table Metadata
Source: Hospital database (CareVue and MetaVision combined).

Row Count: 2,083,180 rows.

Primary Key: ROW_ID.

Links to: PATIENTS (SUBJECT_ID), ADMISSIONS (HADM_ID), and CAREGIVERS (CGID).

Column Definitions and Clinical Significance
SUBJECT_ID & HADM_ID: Identifiers for the patient and hospital stay. Note: Some outpatient reports (like ECGs) may not have an HADM_ID.

CHARTDATE & CHARTTIME: CHARTDATE provides the date of the note. CHARTTIME provides the specific time, though it is missing for certain categories like 'Discharge Summary', 'ECG', and 'Echo'.

CATEGORY & DESCRIPTION: Define the type of note (e.g., 'Radiology', 'Nursing', 'Physician', 'Discharge summary'). The DESCRIPTION further specifies the content, such as whether it is a full 'Report' or an 'Addendum'.

TEXT: The core content of the table. It contains the raw, free-text clinical note. This field is typically large and requires Natural Language Processing (NLP) for analysis.

ISERROR: A binary flag ('1') indicating that a clinician has marked the note as containing an error.

Technical Considerations: Natural Language Processing (NLP)
Data Cleaning: The TEXT field often contains de-identification tags (e.g., [**First Name **]), newlines, and specialized medical abbreviations.

Template Redundancy: Some reports (especially Echo) are generated from templates. It is common to find contradictory statements (e.g., "Mild" and "Severe" in the same paragraph) if a caregiver failed to delete the incorrect template option.

Outpatient Data: This table is unique because it includes radiology and cardiology reports for outpatient visits, providing a longitudinal view of the patient's health beyond a specific ICU stay.

Clinical Context: The Discharge Summary
The 'Discharge Summary' is perhaps the most valuable note type for predictive modeling. It provides a comprehensive overview of the entire hospital stay, including the final diagnosis, procedures performed, and the patient's condition upon leaving.

Information that might interest
For the construction of the predictive model, NOTEEVENTS is an advanced source for semantic features. By applying NLP techniques (like sentiment analysis or keyword extraction for comorbidities), we can identify clinical complexities that aren't explicitly captured in structured tables. For instance, a nursing note mentioning "difficult social situation" or "patient confused" can be a powerful predictor of a prolonged LOS (Length of Stay) or a high risk of readmission. Using these notes allows the model to "read" the patient's chart much like a clinician would.

In [16]:
noteevents_cv_df = create_df(f"{MIMIC_DIR}/NOTEEVENTS.csv")

+--------------------+--------------------+-------+----------+---------+---------+-----------------+-----------+----+-------+--------------------+
|              ROW_ID|          SUBJECT_ID|HADM_ID| CHARTDATE|CHARTTIME|STORETIME|         CATEGORY|DESCRIPTION|CGID|ISERROR|                TEXT|
+--------------------+--------------------+-------+----------+---------+---------+-----------------+-----------+----+-------+--------------------+
|                 174|               22532| 167853|2151-08-04|     NULL|     NULL|Discharge summary|     Report|NULL|   NULL|Admission Date:  ...|
|            Service:|                NULL|   NULL|      NULL|     NULL|     NULL|             NULL|       NULL|NULL|   NULL|                NULL|
|           ADDENDUM:|                NULL|   NULL|      NULL|     NULL|     NULL|             NULL|       NULL|NULL|   NULL|                NULL|
|RADIOLOGIC STUDIE...|                NULL|   NULL|      NULL|     NULL|     NULL|             NULL|       NULL|NULL| 

## CAREGIVERS

The CAREGIVERS table provides identity and role information for the healthcare professionals who recorded data or performed procedures. By linking the CGID (Caregiver ID) found in clinical tables (like CHARTEVENTS or PROCEDUREEVENTS_MV) to this table, we can determine if a specific measurement or note was entered by a doctor, nurse, pharmacist, or respiratory therapist.

Table Metadata
Source: CareVue and MetaVision ICU databases.

Row Count: 7,567 rows.

Primary Key: ROW_ID.

Links to: CHARTEVENTS, NOTEEVENTS, LABEVENTS, PROCEDUREEVENTS_MV, and INPUTEVENTS on CGID.

Column Definitions and Clinical Significance
CGID: The unique identifier for each distinct caregiver. It is the bridge between the clinical actions and the professional's role.

LABEL: A short, free-text code defining the caregiver type (e.g., MD for Medical Doctor, RN for Registered Nurse, RRT for Registered Respiratory Therapist).

DESCRIPTION: A more structured and standardized version of the label. While LABEL might have typos (like "M.D." vs "MD"), the DESCRIPTION column is more consistent, containing only a few unique values (e.g., 'Attending', 'Resident', 'Nurse').

Technical Considerations: Data Quality
Free-Text Variations: Because the LABEL field was often filled manually, it requires cleaning (e.g., converting all variations of "Doctor" to a single category) before it can be used effectively in a model.

Database Merging: The CGID is sourced from both CareVue and MetaVision. While rare, the documentation notes that there is a very small possibility of name collisions between different professionals, though they are treated as unique IDs.

Clinical Context: Care Intensity
The diversity of caregivers assigned to a patient is a direct indicator of clinical complexity:

Specialist Involvement: Frequent entries from specialized services (e.g., PharmD for pharmacists or RRT for respiratory therapists) suggest a patient with high-intensity needs.

Staffing Ratios: In research, this table is often used to calculate nursing workloads or to see how the experience level of the staff (e.g., 'Attending' vs. 'Resident') correlates with patient outcomes.

Information that might interest
For the construction of the predictive model, the CAREGIVERS table allows us to create features based on professional diversity. A patient who requires the intervention of five different types of specialists in the first 24 hours (e.g., MD, RN, RRT, Dietitian, and Physical Therapist) likely has a much higher clinical burden than a patient seen only by a nurse and a general doctor. We can engineer a "Caregiver Diversity Score" as a proxy for severity, which can significantly improve our LOS (Length of Stay) predictions.

In [17]:
caregivers_df = create_df(f"{MIMIC_DIR}/CAREGIVERS.csv")

+------+-----+-----+--------------------+
|ROW_ID| CGID|LABEL|         DESCRIPTION|
+------+-----+-----+--------------------+
|  2228|16174|   RO|           Read Only|
|  2229|16175|   RO|           Read Only|
|  2230|16176|  Res|Resident/Fellow/P...|
|  2231|16177|   RO|           Read Only|
|  2232|16178|   RT|         Respiratory|
+------+-----+-----+--------------------+
only showing top 5 rows
root
 |-- ROW_ID: integer (nullable = true)
 |-- CGID: integer (nullable = true)
 |-- LABEL: string (nullable = true)
 |-- DESCRIPTION: string (nullable = true)

Number of rows: 7567


## LABEVENTS

The LABEVENTS table contains all laboratory-based measurements for patients, including both in-hospital stays and outpatient clinic visits. It records biochemical analysis of fluids (blood, urine, cerebrospinal fluid, etc.) processed by the hospital's central laboratory. Because these results come directly from the lab database, they are considered the gold standard for clinical values in MIMIC-III.

Table Metadata
Source: Hospital laboratory database.

Row Count: 27,854,055 rows.

Primary Key: ROW_ID.

Links to: PATIENTS (SUBJECT_ID), ADMISSIONS (HADM_ID), and D_LABITEMS (ITEMID).

Column Definitions and Clinical Significance
ITEMID: Identifier for the specific lab test (e.g., Creatinine, Glucose, White Blood Cell count). This must be joined with D_LABITEMS to get the test name.

CHARTTIME: Crucially, this represents the time the fluid was acquired (the blood draw), not when the result was published. This is the correct timestamp to use for chronological modeling.

VALUE & VALUENUM: VALUE contains the raw result (text), while VALUENUM provides the numeric equivalent for computation.

VALUEUOM: The Unit of Measurement (e.g., mg/dL, mmol/L).

FLAG: A categorical indicator (e.g., 'abnormal') that identifies if a result falls outside the hospital's reference ranges. This is a powerful pre-engineered feature for severity.

Technical Considerations: Ground Truth vs. Bedside
Ground Truth: While some lab values are repeated in CHARTEVENTS for clinical display, the documentation states that if there is a discrepancy, LABEVENTS should be taken as the ground truth.

Missing HADM_ID: Many rows lack an HADM_ID. This indicates "outpatient" data—tests taken during clinic visits when the patient was not admitted to the hospital.

No STORETIME: Unlike bedside charts, lab data is not manually validated by ICU staff in this table, so there is no STORETIME column.

Clinical Context: Organ Function and Acuity
Lab results are the primary indicators of internal organ failure:

Creatinine/BUN: Indicators of renal (kidney) function.

Lactate: A critical marker of tissue perfusion and sepsis severity.

Electrolytes (Sodium, Potassium): Essential for monitoring metabolic stability.

Information that might interest
For the construction of the predictive model, LABEVENTS provides the objective "biochemical state" of the patient. Lab values are often more standardized and less prone to human error than bedside charting. We can use the FLAG column to quickly identify patients with multiple "abnormal" results in the first 24 hours. Features like "Maximum Lactate level" or "Trend in Creatinine" are highly predictive of complications and, consequently, a longer LOS (Length of Stay).

In [18]:
labevents_df = create_df(f"{MIMIC_DIR}/LABEVENTS.csv")

+------+----------+-------+------+-------------------+-----+--------+--------+--------+
|ROW_ID|SUBJECT_ID|HADM_ID|ITEMID|          CHARTTIME|VALUE|VALUENUM|VALUEUOM|    FLAG|
+------+----------+-------+------+-------------------+-----+--------+--------+--------+
|   281|         3|   NULL| 50820|2101-10-12 16:07:00| 7.39|    7.39|   units|    NULL|
|   282|         3|   NULL| 50800|2101-10-12 18:17:00|  ART|    NULL|    NULL|    NULL|
|   283|         3|   NULL| 50802|2101-10-12 18:17:00|   -1|      -1|   mEq/L|    NULL|
|   284|         3|   NULL| 50804|2101-10-12 18:17:00|   22|      22|   mEq/L|    NULL|
|   285|         3|   NULL| 50808|2101-10-12 18:17:00| 0.93|     .93|  mmol/L|abnormal|
+------+----------+-------+------+-------------------+-----+--------+--------+--------+
only showing top 5 rows
root
 |-- ROW_ID: integer (nullable = true)
 |-- SUBJECT_ID: integer (nullable = true)
 |-- HADM_ID: integer (nullable = true)
 |-- ITEMID: integer (nullable = true)
 |-- CHARTTIME: ti

## MICROBIOLOGYEVENTS

The MICROBIOLOGYEVENTS table contains all information regarding microbiology cultures and antibiotic sensitivity tests. When a patient is suspected of having an infection, clinicians take samples (blood, urine, sputum) to see if organisms grow. If an organism is identified, the lab tests which antibiotics can effectively kill it.

Table Metadata
Source: Hospital microbiology database.

Row Count: 631,726 rows.

Primary Key: ROW_ID.

Links to: PATIENTS (SUBJECT_ID), ADMISSIONS (HADM_ID), and D_ITEMS (via SPEC, ORG, and AB item IDs).

Column Definitions and Clinical Significance
SPEC_TYPE_DESC: The type of specimen collected from the patient (e.g., Blood, Urine, Sputum, Catheter Tip).

ORG_NAME: The name of the organism that grew (e.g., Staphylococcus aureus, Escherichia coli). If this is NULL, it means the culture was negative (no growth).

AB_NAME: The name of the antibiotic tested against the identified organism.

INTERPRETATION: The most critical clinical result. It indicates how the organism responded to the antibiotic:

S (Sensitive): The antibiotic is effective.

R (Resistant): The antibiotic will not work.

I (Intermediate): The antibiotic may work at higher doses.

P (Pending): Results are not yet available.

CHARTTIME & CHARTDATE: These represent when the sample was acquired (collected from the patient). Since cultures take days to grow, this timestamp is the clinical "start" of the infection investigation.

Technical Considerations: Culture Timelines
Growth Reporting: Microbiological data is slow. While the CHARTTIME reflects the collection, the actual result (the organism name and sensitivity) only becomes known to clinicians 24 to 72 hours later.

Null Organisms: A row with a null ORG_NAME is just as important as a positive one—it confirms the absence of a detectable infection in that specific sample.

Clinical Context: Sepsis and Resistance
This table is the primary source for studying Sepsis, one of the leading causes of death in the ICU.

Multi-Drug Resistance (MDR): Patients with organisms marked as "R" (Resistant) to multiple common antibiotics often require specialized, expensive treatments and longer recovery times.

Source of Infection: The SPEC_TYPE_DESC helps clinicians determine if an infection is primary (e.g., Pneumonia via sputum) or healthcare-associated (e.g., Catheter-related via blood).

Information that might interest
For the construction of the predictive model, MICROBIOLOGYEVENTS provides indicators of infectious complexity. A positive culture (ORG_NAME is not null) or the presence of antibiotic-resistant bacteria (INTERPRETATION = 'R') are strong predictors of a significantly extended LOS (Length of Stay). We can use this table to create features such as "Confirmed Infection in first 48h" or "Presence of Drug-Resistant Organism," which provide the model with a clear reason for prolonged intensive care and complex antibiotic regimens.

In [19]:
microbiologyevents_df = create_df(f"{MIMIC_DIR}/MICROBIOLOGYEVENTS.csv")

+------+----------+-------+-------------------+-------------------+-----------+--------------------+----------+--------------------+-----------+---------+-------+-------------+-------------------+--------------+--------------+
|ROW_ID|SUBJECT_ID|HADM_ID|          CHARTDATE|          CHARTTIME|SPEC_ITEMID|      SPEC_TYPE_DESC|ORG_ITEMID|            ORG_NAME|ISOLATE_NUM|AB_ITEMID|AB_NAME|DILUTION_TEXT|DILUTION_COMPARISON|DILUTION_VALUE|INTERPRETATION|
+------+----------+-------+-------------------+-------------------+-----------+--------------------+----------+--------------------+-----------+---------+-------+-------------+-------------------+--------------+--------------+
|   744|        96| 170324|2156-04-13 00:00:00|2156-04-13 14:18:00|      70021|BRONCHOALVEOLAR L...|     80026|PSEUDOMONAS AERUG...|          1|     NULL|   NULL|         NULL|               NULL|          NULL|          NULL|
|   745|        96| 170324|2156-04-20 00:00:00|2156-04-20 13:10:00|      70062|             

## PRESCRIPTIONS

The PRESCRIPTIONS table contains all medication-related order entries from the hospital’s provider order entry (POE) system. While INPUTEVENTS records exactly what was infused at the bedside, PRESCRIPTIONS provides a broader view of the clinical intent, including oral medications, scheduled antibiotics, and drugs prescribed both inside and outside the ICU.

Table Metadata
Source: Hospital provider order entry (POE) database.

Row Count: 4,156,450 rows.

Primary Key: ROW_ID.

Links to: PATIENTS (SUBJECT_ID), ADMISSIONS (HADM_ID), and ICUSTAYS (ICUSTAY_ID).

Column Definitions and Clinical Significance
STARTDATE & ENDDATE: The specific date range during which the prescription was active and valid.

DRUG, DRUG_NAME_POE & DRUG_NAME_GENERIC: Various naming conventions for the prescribed drug. DRUG_NAME_GENERIC is often the most useful for standardizing data.

GSN & NDC: National drug coding systems (Generic Sequence Number and National Drug Code). These allow for precise identification of medications regardless of brand names.

DOSE_VAL_RX & DOSE_UNIT_RX: The amount of the drug prescribed (e.g., 500) and its unit (e.g., mg).

ROUTE: The method of administration (e.g., PO for oral, IV for intravenous, SC for subcutaneous).

DRUG_TYPE: Categorizes the medication, helping to distinguish between main treatments and additives.

Technical Considerations: Intent vs. Administration
Prescription vs. Delivery: Not all prescriptions result in administration. A doctor may prescribe a drug "as needed" (PRN), but the patient may never require it.

Cancellation Data: A critical note for MIMIC-III v1.0 is that this table does not explicitly specify if an order was later cancelled.

Coverage: This table includes pharmacy orders for the entire hospital stay, offering a more complete pharmacological profile than the ICU-centric INPUTEVENTS.

Clinical Context: Polypharmacy and Complexity
The variety and volume of prescriptions are direct indicators of a patient's medical complexity:

Antibiotic Escalation: Moving from narrow-spectrum to broad-spectrum antibiotics often indicates a worsening infection.

Polypharmacy: A high number of concurrent prescriptions is a known risk factor for adverse drug events and longer recovery times.

Chronic vs. Acute: By looking at the drugs, we can often infer a patient's pre-existing conditions (comorbidities) even if they aren't explicitly listed in the diagnosis table.

Information that might interest
For the construction of the predictive model, PRESCRIPTIONS provides excellent static and categorical features. Identifying specific drug classes prescribed in the first 24 hours—such as insulin for glucose control or anticoagulants for prophylaxis—can help the model categorize the patient's risk level. The "Total count of active prescriptions" at the start of the stay is a simple yet effective proxy for clinical complexity and is highly correlated with a patient's LOS (Length of Stay).

In [20]:
prescriptions_df = create_df(f"{MIMIC_DIR}/PRESCRIPTIONS.csv")

+-------+----------+-------+----------+-------------------+-------------------+---------+--------------+-------------+-----------------+-----------------+------+---------+--------------------+-----------+------------+-------------+--------------+-----+
| ROW_ID|SUBJECT_ID|HADM_ID|ICUSTAY_ID|          STARTDATE|            ENDDATE|DRUG_TYPE|          DRUG|DRUG_NAME_POE|DRUG_NAME_GENERIC|FORMULARY_DRUG_CD|   GSN|      NDC|       PROD_STRENGTH|DOSE_VAL_RX|DOSE_UNIT_RX|FORM_VAL_DISP|FORM_UNIT_DISP|ROUTE|
+-------+----------+-------+----------+-------------------+-------------------+---------+--------------+-------------+-----------------+-----------------+------+---------+--------------------+-----------+------------+-------------+--------------+-----+
|2214776|         6| 107064|      NULL|2175-06-11 00:00:00|2175-06-12 00:00:00|     MAIN|    Tacrolimus|   Tacrolimus|       Tacrolimus|            TACR1|021796|469061711|         1mg Capsule|          2|          mg|            2|          

## DIAGNOSES_ICD

The DIAGNOSES_ICD table contains the official International Classification of Diseases (ICD-9) codes assigned to patients for each hospital stay. These codes are the standard for clinical labeling, research, and hospital billing, providing a structured way to identify the primary reason for admission as well as any underlying chronic conditions (comorbidities).

Table Metadata
Source: Hospital administrative and billing database.

Row Count: 651,047 rows.

Primary Key: ROW_ID.

Links to: PATIENTS (SUBJECT_ID), ADMISSIONS (HADM_ID), and D_ICD_DIAGNOSES (on ICD9_CODE).

Column Definitions and Clinical Significance
SUBJECT_ID & HADM_ID: Identifiers to link the diagnosis to the patient and the specific hospital stay.

ICD9_CODE: The actual diagnosis code (e.g., 4019 for Hypertension, 25000 for Diabetes). Note that in MIMIC-III, all codes are based on the ICD-9-CM system.

SEQ_NUM: This is the "Priority" or "Sequence" of the diagnosis.

SEQ_NUM = 1 is almost always the Primary Diagnosis (the main reason for the hospital stay).

Higher numbers represent secondary diagnoses or complications that arose during the stay.

Technical Considerations: The Billing Perspective
Retrospective Nature: A critical point is that ICD codes are generated at the end of the hospital stay for billing purposes. They are not recorded in real-time as the patient is being treated.

Format: The decimal point is implied in the database. For most codes, it falls between the 3rd and 4th digit (e.g., 410.01 is stored as 41001). For "V" codes (supplementary factors), the decimal is between the 2nd and 3rd digit.

Clinical Context: Comorbidities and Severity
The list of diagnoses allows for the calculation of complexity scores:

Elixhauser Comorbidity Index: A common method in research to use these ICD codes to measure how "sick" a patient was before the current acute event.

Primary vs. Secondary: The difference between a primary diagnosis (e.g., Acute Myocardial Infarction) and a secondary one (e.g., Chronic Kidney Disease) is vital for understanding why a patient might stay longer in the ICU than expected.

Information that might interest
For the construction of the predictive model, DIAGNOSES_ICD provides the ground truth for clinical labels. While you can't use all these codes as "input features" (since they are only known after the stay), you can use them to group patients into categories (e.g., "All Respiratory Patients") or to identify pre-existing conditions that were known upon admission. The SEQ_NUM is particularly useful for focusing only on the most severe condition affecting the patient, which has a massive impact on the expected LOS (Length of Stay).

In [21]:
diagnoses_icd_df = create_df(f"{MIMIC_DIR}/DIAGNOSES_ICD.csv")

+------+----------+-------+-------+---------+
|ROW_ID|SUBJECT_ID|HADM_ID|SEQ_NUM|ICD9_CODE|
+------+----------+-------+-------+---------+
|  1297|       109| 172335|      1|    40301|
|  1298|       109| 172335|      2|      486|
|  1299|       109| 172335|      3|    58281|
|  1300|       109| 172335|      4|     5855|
|  1301|       109| 172335|      5|     4254|
+------+----------+-------+-------+---------+
only showing top 5 rows
root
 |-- ROW_ID: integer (nullable = true)
 |-- SUBJECT_ID: integer (nullable = true)
 |-- HADM_ID: integer (nullable = true)
 |-- SEQ_NUM: integer (nullable = true)
 |-- ICD9_CODE: string (nullable = true)

Number of rows: 651047


## PROCEDURES_ICD

The PROCEDURES_ICD table contains the International Classification of Diseases (ICD-9-CM) codes for procedures performed during a patient's hospital stay. While the PROCEDUREEVENTS_MV table captures bedside ICU interventions in real-time, this table provides the high-level administrative record of major operations and diagnostic procedures (like appendectomies, bypass surgeries, or endoscopies) that occurred at any point during the hospitalization.

Table Metadata
Source: Hospital administrative and billing database.

Row Count: 240,095 rows.

Primary Key: ROW_ID.

Links to: PATIENTS (SUBJECT_ID), ADMISSIONS (HADM_ID), and D_ICD_PROCEDURES (on ICD9_CODE).

Column Definitions and Clinical Significance
SUBJECT_ID & HADM_ID: Identifiers linking the procedure to the specific patient and hospital admission.

ICD9_CODE: The standardized ICD-9-CM procedure code. To understand the specific procedure (e.g., 38.0 for incision of vessel), this must be joined with the D_ICD_PROCEDURES dictionary.

SEQ_NUM: The order in which the procedures are listed. Similar to diagnoses, the first sequence usually represents the "Principal Procedure," often the one most closely related to the primary diagnosis and resource consumption.

Technical Considerations: Procedures vs. Events
Billing Focus: Like the diagnoses table, these codes are generated retrospectively after the patient is discharged. They are recorded for every hospitalization in the MIMIC-III dataset.

Scope: This table includes major surgeries performed in the Operating Room (OR) which might not appear in the ICU-specific PROCEDUREEVENTS tables.

ICD-9 Standard: All procedure codes in this version of MIMIC are ICD-9 based.

Clinical Context: Surgical Complexity
Procedures recorded here are often the primary drivers of hospital resource utilization:

Invasive vs. Non-Invasive: The presence of high-intensity surgical codes (e.g., cardiac or neurosurgical procedures) is a strong indicator of a patient's clinical path.

Post-Operative Recovery: Knowing a patient underwent a major procedure allows researchers to distinguish between "Medical" admissions and "Post-Surgical" admissions, which have very different expected outcomes.

Information that might interest
For the construction of the predictive model, PROCEDURES_ICD is essential for feature validation and cohort definition. Although these codes are technically "post-hoc," the fact that a patient underwent a specific major surgery (like a coronary artery bypass) is usually known early in the stay. By using these codes, we can create categorical features such as "Surgical Patient Type" or "Number of Major Procedures," which are heavily correlated with the overall LOS (Length of Stay). It helps the model distinguish between a patient staying for observation and one staying for complex surgical recovery.

In [22]:
procedures_icd_df = create_df(f"{MIMIC_DIR}/PROCEDURES_ICD.csv")

+------+----------+-------+-------+---------+
|ROW_ID|SUBJECT_ID|HADM_ID|SEQ_NUM|ICD9_CODE|
+------+----------+-------+-------+---------+
|   944|     62641| 154460|      3|     3404|
|   945|      2592| 130856|      1|     9671|
|   946|      2592| 130856|      2|     3893|
|   947|     55357| 119355|      1|     9672|
|   948|     55357| 119355|      2|      331|
+------+----------+-------+-------+---------+
only showing top 5 rows
root
 |-- ROW_ID: integer (nullable = true)
 |-- SUBJECT_ID: integer (nullable = true)
 |-- HADM_ID: integer (nullable = true)
 |-- SEQ_NUM: integer (nullable = true)
 |-- ICD9_CODE: integer (nullable = true)

Number of rows: 240095


## D_ITEMS

The D_ITEMS table is the primary dictionary for all measurements and events recorded within the ICU databases (CareVue and MetaVision). Every time a heart rate is charted or a medication is infused, it is associated with an ITEMID. This table maps those IDs to human-readable labels, abbreviations, and categories.

Table Metadata
Source: CareVue and MetaVision ICU databases.

Row Count: 12,487 rows.

Primary Key: ITEMID (Alternate Primary Key).

Links to: CHARTEVENTS, DATETIMEEVENTS, INPUTEVENTS_CV/MV, OUTPUTEVENTS, MICROBIOLOGYEVENTS, and PROCEDUREEVENTS_MV.

Column Definitions and Clinical Significance
ITEMID: The unique identifier for a clinical concept.

LABEL & ABBREVIATION: The descriptive name of the item (e.g., "Heart Rate") and its common abbreviation (e.g., "HR"). Abbreviations are primarily available for MetaVision data.

DBSOURCE: Indicates if the ID came from 'carevue' or 'metavision'.

Crucial Note: MetaVision ITEMIDs are always > 220,000.

LINKSTO: Specifies which event table contains the data for this item (e.g., if LINKSTO is 'chartevents', you will find the actual measurements in the CHARTEVENTS table).

CATEGORY: Groups items by clinical type, such as 'Respiratory', 'Hemodynamics', or 'Medications'.

PARAM_TYPE: Defines the nature of the data: numeric, date, or text.

Technical Considerations: The Challenge of Duplication
The biggest hurdle when using D_ITEMS is that the same clinical concept often has multiple IDs:

System Split: Heart Rate is ITEMID 211 in CareVue but ITEMID 220045 in MetaVision.

Free-Text Entry: In CareVue, manual entries led to synonyms and misspellings (e.g., "Weight", "Weight (kg)", and "Wgt") having different ITEMIDs.

Best Practice: Always search for all possible labels and abbreviations to ensure you capture the complete history of a physiological parameter across the entire patient population.

Clinical Context: Dictionary Mapping
This table does not link to LABEVENTS. Lab results have their own separate dictionary called D_LABITEMS. D_ITEMS is strictly for data captured within the ICU environment by bedside monitors and staff.

Information that might interest
For the construction of the predictive model, D_ITEMS is the Rosetta Stone of your feature engineering process. You cannot extract "Blood Pressure" or "Oxygen Saturation" without first querying this table to find every relevant ITEMID. For a robust LOS (Length of Stay) prediction, you will need to create a mapping dictionary that groups these disparate IDs into consolidated clinical features (e.g., mapping both 211 and 220045 to a single "heart_rate" column). This step is vital to ensure your model treats data from different hospital eras consistently.

In [23]:
d_items_df = create_df(f"{MIMIC_DIR}/D_ITEMS.csv")

+------+------+--------------------+------------+--------+-----------+--------+--------+----------+---------+
|ROW_ID|ITEMID|               LABEL|ABBREVIATION|DBSOURCE|    LINKSTO|CATEGORY|UNITNAME|PARAM_TYPE|CONCEPTID|
+------+------+--------------------+------------+--------+-----------+--------+--------+----------+---------+
|   457|   497|Patient controlle...|        NULL| carevue|chartevents|    NULL|    NULL|      NULL|     NULL|
|   458|   498|   PCA Lockout (Min)|        NULL| carevue|chartevents|    NULL|    NULL|      NULL|     NULL|
|   459|   499|      PCA Medication|        NULL| carevue|chartevents|    NULL|    NULL|      NULL|     NULL|
|   460|   500|      PCA Total Dose|        NULL| carevue|chartevents|    NULL|    NULL|      NULL|     NULL|
|   461|   501|  PCV Exh Vt (Obser)|        NULL| carevue|chartevents|    NULL|    NULL|      NULL|     NULL|
+------+------+--------------------+------------+--------+-----------+--------+--------+----------+---------+
only showi

## D_LABITEMS

The D_LABITEMS table is the specialized dictionary for all laboratory measurements in the MIMIC-III database. While D_ITEMS covers bedside ICU data, D_LABITEMS defines the codes for biochemistry, hematology, and other lab tests processed by the hospital's central laboratory. Because these data come from a unified hospital system, they are remarkably consistent across all years of the dataset.

Table Metadata
Source: Hospital laboratory database.

Row Count: 753 rows.

Primary Key: ITEMID (Unique for lab concepts).

Links to: LABEVENTS on ITEMID.

Column Definitions and Clinical Significance
ITEMID: The unique identifier for a specific laboratory concept.

LABEL: The name of the test (e.g., "Creatinine", "White Blood Cells", "Lactate").

FLUID: Specifies the source of the sample. This is a critical discriminator; for example, Glucose measured in BLOOD has a different clinical meaning and reference range than Glucose measured in URINE or CSF (Cerebrospinal Fluid).

CATEGORY: High-level grouping of tests, such as 'Blood Gas', 'Chemistry', or 'Hematology'.

LOINC_CODE: The Logical Observation Identifiers Names and Codes (LOINC). This is a universal standard for identifying medical laboratory observations. Mapping to LOINC allows researchers to easily compare MIMIC data with other clinical datasets worldwide.

Technical Considerations: Consistency and Standards
Single ID per Concept: Unlike the ICU bedside data (where Heart Rate has multiple IDs), laboratory data usually has a one-to-one mapping. One ITEMID typically represents one specific test/fluid combination consistently from 2001 to 2012.

Post-hoc LOINC: Most LOINC codes were assigned after the data collection to help standardize the database. While very helpful, they may not be present for every single row.

Separation of Concerns: Lab items are kept in this separate table (instead of D_ITEMS) because they originate from a different administrative system and follow different naming conventions.

Clinical Context: The Fluid Factor
In predictive modeling, the FLUID column is essential for data cleaning. Analyzing "pH" without checking if it comes from Blood (Arterial Blood Gas) or Urine would lead to incorrect clinical features. This table ensures that you are comparing "apples to apples" when building physiological profiles.

Information that might interest
For the construction of the predictive model, D_LABITEMS is your guide for feature selection. You will use this table to identify the ITEMIDs for the most predictive biomarkers, such as Troponin (for cardiac events), Bilirubin (for liver function), or Hemoglobin (for anemia/bleeding). By filtering by FLUID = 'BLOOD' and CATEGORY = 'CHEMISTRY', you can quickly isolate the core metabolic markers that significantly influence a patient's LOS (Length of Stay).

In [24]:
d_labitems_df = create_df(f"{MIMIC_DIR}/D_LABITEMS.csv")

+------+------+--------------------+--------------------+----------+----------+
|ROW_ID|ITEMID|               LABEL|               FLUID|  CATEGORY|LOINC_CODE|
+------+------+--------------------+--------------------+----------+----------+
|   546| 51346|              Blasts|Cerebrospinal Flu...|Hematology|   26447-3|
|   547| 51347|         Eosinophils|Cerebrospinal Flu...|Hematology|   26451-5|
|   548| 51348|     Hematocrit, CSF|Cerebrospinal Flu...|Hematology|   30398-2|
|   549| 51349|Hypersegmented Ne...|Cerebrospinal Flu...|Hematology|   26506-6|
|   550| 51350|   Immunophenotyping|Cerebrospinal Flu...|Hematology|      NULL|
+------+------+--------------------+--------------------+----------+----------+
only showing top 5 rows
root
 |-- ROW_ID: integer (nullable = true)
 |-- ITEMID: integer (nullable = true)
 |-- LABEL: string (nullable = true)
 |-- FLUID: string (nullable = true)
 |-- CATEGORY: string (nullable = true)
 |-- LOINC_CODE: string (nullable = true)

Number of rows: 

## D_ICD_DIAGNOSES

The D_ICD_DIAGNOSES table serves as the definitive dictionary for all International Classification of Diseases, 9th Revision (ICD-9) diagnosis codes used in the MIMIC-III database. While the DIAGNOSES_ICD table records which codes were assigned to a patient, this table provides the official titles and descriptions for those codes.

Table Metadata
Source: Online ICD-9-CM coding resources.

Row Count: 14,567 rows.

Primary Key: ICD9_CODE.

Links to: DIAGNOSES_ICD on ICD9_CODE.

Column Definitions and Clinical Significance
ICD9_CODE: The unique ICD-9 identifier for a specific medical condition. These codes are structured hierarchically (e.g., codes starting with '410' relate to Myocardial Infarction).

SHORT_TITLE: A concise, abbreviated version of the diagnosis (e.g., "Infect endocardit NOS").

LONG_TITLE: The full, formal clinical description of the diagnosis (e.g., "Acute and subacute bacterial endocarditis"). This is the most accurate field for identifying specific patient cohorts.

Technical Considerations: ICD-9 Structure
Implicit Decimals: As noted in the DIAGNOSES_ICD analysis, the decimal points are not stored in the database. For example, 507.0 (Pneumonitis due to inhalation of food or vomit) is stored as 5070.

V and E Codes: This table also includes "V" codes (factors influencing health status, like vaccinations or screenings) and "E" codes (external causes of injury, like a motor vehicle accident).

Clinical Context: High-Level Aggregation
Because there are over 14,000 unique ICD-9 codes, researchers often use this table to group specific codes into broader "Clinical Classifications" (CCS) or "Comorbidity Groups." For example, dozens of different ICD-9 codes for various heart valve issues might be rolled up into a single "Valvular Disease" feature for a machine learning model.

Information that might interest
For the construction of the predictive model, D_ICD_DIAGNOSES is vital for feature interpretability. When the model identifies a specific ICD9_CODE as a top predictor for a long LOS (Length of Stay), we use this table to explain why (e.g., "The model prioritized 'Septicemia' as a key driver of long stays"). It also allows us to filter the dataset for specific sub-populations, such as "all patients with chronic kidney disease," by searching for relevant keywords within the LONG_TITLE column.

In [25]:
d_icd_diagnoses_df = create_df(f"{MIMIC_DIR}/D_ICD_DIAGNOSES.csv")

+------+---------+--------------------+--------------------+
|ROW_ID|ICD9_CODE|         SHORT_TITLE|          LONG_TITLE|
+------+---------+--------------------+--------------------+
|   174|    01166|TB pneumonia-oth ...|Tuberculous pneum...|
|   175|    01170|TB pneumothorax-u...|Tuberculous pneum...|
|   176|    01171|TB pneumothorax-n...|Tuberculous pneum...|
|   177|    01172|TB pneumothorx-ex...|Tuberculous pneum...|
|   178|    01173|TB pneumothorax-m...|Tuberculous pneum...|
+------+---------+--------------------+--------------------+
only showing top 5 rows
root
 |-- ROW_ID: integer (nullable = true)
 |-- ICD9_CODE: string (nullable = true)
 |-- SHORT_TITLE: string (nullable = true)
 |-- LONG_TITLE: string (nullable = true)

Number of rows: 14567


## D_ICD_PROCEDURES

The D_ICD_PROCEDURES table is the final reference dictionary, providing the definitions for International Classification of Diseases, 9th Revision (ICD-9) procedure codes. While D_ICD_DIAGNOSES tells us what the patient had, this table tells us exactly what major interventions were performed (e.g., surgeries, biopsies, or specialized imaging) during the entire hospital stay.

Table Metadata
Source: Online ICD-9-CM procedure coding resources.

Row Count: 3,882 rows.

Primary Key: ICD9_CODE.

Links to: PROCEDURES_ICD on ICD9_CODE.

Column Definitions and Clinical Significance
ICD9_CODE: The standardized procedure identifier. Unlike diagnosis codes, these are usually shorter (often 2 to 4 digits).

SHORT_TITLE: An abbreviated version of the procedure name (e.g., "Aortocoronary bypass").

LONG_TITLE: The full clinical description of the procedure (e.g., "Aortocoronary bypass for heart revascularization").

Technical Considerations: Procedures vs. Events
Billing Context: Like the diagnosis codes, these are assigned by hospital coders after discharge. They are optimized for billing and high-level clinical reporting rather than minute-by-minute ICU tracking.

ICD-9-CM Volume 3: These codes follow a different structure than the diagnosis codes. For example, code 38.0 refers to an incision of a vessel, whereas in the diagnosis table, 380 would refer to something entirely different (a disorder of the external ear).

Clinical Context: Identifying Surgical Cohorts
This table is the primary tool for identifying "Surgical" vs. "Medical" patients:

Surgical Indicators: Any procedure code within the ranges dedicated to operations (typically codes 01 through 86) indicates that the patient underwent an invasive procedure in the operating room.

Complexity Mapping: You can use keywords in the LONG_TITLE to flag high-complexity procedures—such as organ transplants or open-heart surgeries—that fundamentally alter the expected LOS (Length of Stay) profile.

Information that might interest
For the construction of the predictive model, D_ICD_PROCEDURES is essential for feature engineering via grouping. With nearly 4,000 codes, it's often more effective to group them into "Surgical Categories" (e.g., Cardiovascular, Orthopedic, or Digestive). By linking this table to PROCEDURES_ICD, you can create a feature like "Total Major Surgeries," which acts as a proxy for both surgical risk and recovery time. This allows the model to differentiate a patient in the ICU for routine post-operative monitoring from one who had a multi-stage emergency surgery.

In [26]:
d_icd_procedures_df = create_df(f"{MIMIC_DIR}/D_ICD_PROCEDURES.csv")

+------+---------+--------------------+--------------------+
|ROW_ID|ICD9_CODE|         SHORT_TITLE|          LONG_TITLE|
+------+---------+--------------------+--------------------+
|   264|      851|          Canthotomy|          Canthotomy|
|   265|      852|     Blepharorrhaphy|     Blepharorrhaphy|
|   266|      859|Adjust lid positi...|Other adjustment ...|
|   267|      861|Lid reconst w ski...|Reconstruction of...|
|   268|      862|Lid reconst w muc...|Reconstruction of...|
+------+---------+--------------------+--------------------+
only showing top 5 rows
root
 |-- ROW_ID: integer (nullable = true)
 |-- ICD9_CODE: integer (nullable = true)
 |-- SHORT_TITLE: string (nullable = true)
 |-- LONG_TITLE: string (nullable = true)

Number of rows: 3882
